# 08 · RL state design — discrete signal contexts

**State** s = context k(x) of the recording, built only from label-free information:

s = [SNR, log-energy, variance, spectral centroid, bandwidth, flatness, spectral entropy, ZCR,
kurtosis, duration, cepstral heart rate & prominence, 32-band log-PSD] → robust scaling →
**PCA (6 components, whitened)** + one-hot **auscultation location** → **k-means** → k ∈ {0..K−1}.

Tabular methods need a finite state set; K trades adaptivity (more contexts, more specific
configurations) against estimation noise (fewer recordings per context). K = 1 means "one
global configuration" — the non-adaptive special case. K is chosen on the **validation split**
using the DP-optimal policy of each K (Notebook 10 reuses the chosen K).

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
import matplotlib.pyplot as plt
import seaborn as sns

meta = pd.read_csv(P["processed"] / "proxy_records.csv")
qc = pd.read_csv(P["metadata"] / "records_qc.csv")
allrec = qc[qc.qc_keep].reset_index(drop=True)
train_df = allrec[allrec.record_id.isin(meta.record_id[meta.split == "train"])]

In [ ]:
Ks = [1, 2, 3, 4, 6, 8, 10, 12]
ctx_cols = {"record_id": allrec.record_id}
for K in Ks:
    cm = ContextModel(K, CFG["context"]["n_pca"], seed=CFG["seed"]).fit(train_df)
    ctx_cols[f"ctx_K{K}"] = cm.predict(allrec)
ctx = pd.DataFrame(ctx_cols)
ctx.to_csv(P["processed"] / "contexts.csv", index=False)

rows = []
for K in Ks:
    prob = RLProblem(CFG, P, K=K)
    mdp = prob.mdp("train")
    dp = value_iteration(mdp)
    mv = prob.evaluate(dp["policy"], "val")
    mt = prob.evaluate(dp["policy"], "train")
    rows.append({"K": K, "train_value (DP optimum)": mdp.policy_value(dp["policy"]), "train_macro_f1": mt["macro_f1"],
                 "val_macro_f1": mv["macro_f1"], "val_bal_acc": mv["balanced_acc"], "val_auroc": mv["auroc"],
                 "val_mean_reward": mv["mean_reward"], "distinct configs": len(set(dp["policy"]))})
ktab = pd.DataFrame(rows)
ktab.to_csv(P["tables"] / "08_context_count_selection.csv", index=False)
ktab

In [ ]:
best_K = int(ktab.loc[ktab.val_mean_reward.idxmax(), "K"])
print("chosen K (max validation reward):", best_K)
print(f"validation reward: K={best_K}: {ktab.val_mean_reward.max():.4f} vs K=1: {ktab.val_mean_reward[ktab.K == 1].item():.4f}")
save_json({"K": best_K, "criterion": "validation mean reward of the DP-optimal policy"}, P["results"] / "chosen_K.json")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(ktab.K, ktab["train_value (DP optimum)"], "o-", label="train (optimistic: argmax over 2,100)")
ax[0].plot(ktab.K, ktab.val_mean_reward, "o-", label="validation")
ax[0].set_xlabel("number of contexts K"); ax[0].set_ylabel("mean Δ-reward"); ax[0].legend(); ax[0].axvline(best_K, ls=":", c="k")
ax[0].set_title("Adaptivity vs. estimation noise (winner's curse)")
ax[1].plot(ktab.K, ktab.val_macro_f1, "o-", label="val macro-F1"); ax[1].plot(ktab.K, ktab.val_bal_acc, "o-", label="val balanced acc")
ax[1].legend(); ax[1].set_xlabel("K")
plt.tight_layout(); savefig(fig, "08_choose_K"); plt.show()

In [ ]:
cm = ContextModel(best_K, CFG["context"]["n_pca"], seed=CFG["seed"]).fit(train_df)
Z = cm.embed(allrec)
lab = allrec.merge(ctx[["record_id", f"ctx_K{best_K}"]], on="record_id")
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=lab[f"ctx_K{best_K}"], palette="tab10", s=6, ax=ax[0]); ax[0].set_title("contexts in PC1–PC2")
pd.crosstab(lab[f"ctx_K{best_K}"], lab.location, normalize="index").plot.bar(stacked=True, ax=ax[1]); ax[1].set_title("location mix per context")
m = lab[lab.murmur_label.isin(["Present", "Absent"])]
m.groupby(f"ctx_K{best_K}").murmur_label.apply(lambda s: (s == "Present").mean()).plot.bar(ax=ax[2]); ax[2].set_title("murmur prevalence per context")
plt.tight_layout(); savefig(fig, "08_contexts"); plt.show()
lab.groupby(f"ctx_K{best_K}")[["snr_db", "centroid", "bandwidth", "ceps_bpm", "proc_duration_s"]].median().round(1)